# InceptionV1 (GoogLeNet) DPU Benchmark
Measures inference speed and power on the KV260 **FPGA DPU (B512)**.

InceptionV1 is an **image classifier** — same task as ResNet50 but different architecture.
It introduced the 'Inception module' — parallel convolutions at multiple scales.

**Run from Jupyter on the KV260** (`http://192.168.68.60:9090/lab`)

In [ ]:
import os, time, numpy as np

POWER_SENSOR = "/sys/class/hwmon/hwmon2/power1_input"
N_WARMUP = 10
N_BENCHMARK = 100

# Smart path: check local models/ first, then pynq default
NOTEBOOK_DIR = os.path.dirname(os.path.abspath("dpu_bench.ipynb"))
XMODEL_LOCAL = os.path.join(NOTEBOOK_DIR, "models", "dpu_tf_inceptionv1.xmodel")
XMODEL_PYNQ  = "/root/jupyter_notebooks/pynq-dpu/dpu_tf_inceptionv1.xmodel"
XMODEL = XMODEL_LOCAL if os.path.exists(XMODEL_LOCAL) else XMODEL_PYNQ

def read_power_mw():
    with open(POWER_SENSOR) as f:
        return int(f.read().strip()) / 1000

print(f"Using xmodel: {XMODEL}")
print(f"Current board power: {read_power_mw()/1000:.2f} W (idle)")

## Step 1 — Load DPU Overlay

In [ ]:
from pynq_dpu import DpuOverlay
overlay = DpuOverlay("dpu.bit")
print("DPU overlay loaded!")

## Step 2 — Load InceptionV1 Model

In [ ]:
overlay.load_model(XMODEL)
dpu = overlay.runner

in_t  = dpu.get_input_tensors()
out_t = dpu.get_output_tensors()
print(f"Input shape:  {in_t[0].dims}")
print(f"Output shape: {out_t[0].dims}  (1001 classes — ImageNet + background)")

## Step 3 — Warmup + Benchmark

In [ ]:
in_d  = [np.zeros(t.dims, dtype=np.int8) for t in in_t]
out_d = [np.zeros(t.dims, dtype=np.int8) for t in out_t]

for _ in range(N_WARMUP):
    job = dpu.execute_async(in_d, out_d)
    dpu.wait(job)
print("Warmup done!")

power_samples = []
start = time.time()
for i in range(N_BENCHMARK):
    job = dpu.execute_async(in_d, out_d)
    dpu.wait(job)
    if i % 10 == 0:
        power_samples.append(read_power_mw())
elapsed = time.time() - start

avg_power_w = sum(power_samples) / len(power_samples) / 1000
fps = N_BENCHMARK / elapsed
latency_ms = elapsed / N_BENCHMARK * 1000
fps_per_watt = fps / avg_power_w

print("=" * 40)
print("INCEPTIONV1 DPU BENCHMARK RESULTS")
print("=" * 40)
print(f"Platform:    KV260 DPU (B512)")
print(f"FPS:         {fps:.1f}")
print(f"Latency:     {latency_ms:.1f} ms/frame")
print(f"Power:       {avg_power_w:.2f} W")
print(f"FPS/Watt:    {fps_per_watt:.2f}")
print("=" * 40)

del overlay

In [ ]:
import cv2, json, glob, numpy as np
from pynq_dpu import DpuOverlay

# Load ImageNet labels
label_files = glob.glob("/home/ubuntu/**/labels.json", recursive=True) + \
              glob.glob("/home/ubuntu/imagenet_labels.json")
label_files = [f for f in label_files if os.path.exists(f)]
if not label_files:
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/anishathalye/imagenet-simple-labels/master/imagenet-simple-labels.json",
        "/home/ubuntu/imagenet_labels.json")
    label_files = ["/home/ubuntu/imagenet_labels.json"]
with open(label_files[0]) as f:
    labels = json.load(f)

# Load pre-resized fish image (224x224 — same as ResNet50)
IMAGE_PATHS = [
    "/root/jupyter_notebooks/dpu_benchmark/test_image_224.jpg",
    "/home/ubuntu/test_image_224.jpg",
]
img_path = next((p for p in IMAGE_PATHS if os.path.exists(p)), None)
img = cv2.imread(img_path)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# Normalize for InceptionV1 (same as ResNet50)
mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
std  = np.array([0.229, 0.224, 0.225], dtype=np.float32)
img_norm = (img_rgb.astype(np.float32) / 255.0 - mean) / std
img_int8 = (img_norm * 128).clip(-128, 127).astype(np.int8)

# Run inference
overlay3 = DpuOverlay("dpu.bit")
overlay3.load_model(XMODEL)
dpu3 = overlay3.runner
in_t3  = dpu3.get_input_tensors()
out_t3 = dpu3.get_output_tensors()
in_d3  = [np.zeros(t.dims, dtype=np.int8) for t in in_t3]
out_d3 = [np.zeros(t.dims, dtype=np.int8) for t in out_t3]

in_d3[0][0] = img_int8
job = dpu3.execute_async(in_d3, out_d3)
dpu3.wait(job)

scores = out_d3[0][0].astype(np.float32)
# InceptionV1 has 1001 outputs (background + 1000 classes)
top5_idx = np.argsort(scores)[::-1][:5]

print(f"Image: fish.jpg (224x224)")
print("InceptionV1 DPU predicts:")
for rank, idx in enumerate(top5_idx):
    # offset by 1 for background class
    label_idx = idx - 1 if idx > 0 else 0
    label = labels[label_idx] if label_idx < len(labels) else f"class_{idx}"
    print(f"  {rank+1}. {label} (score: {scores[idx]:.1f})")

del overlay3

## Bonus — What Does InceptionV1 See? (Real Image Test)